In [2]:
# --- 1. Import libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ML models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

# --- 2. Load your dataset ---
df = pd.read_csv("Air Traffic Data Cor Updated.csv")
# Ensure 'Date' is a datetime column
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# --- 3. Define target columns ---
target_cols = [
    'domestic passengers',
    'international passengers',
    'domestic freight(in tonne)',
    'international freight(in tonne)'
]

# --- 4. Function to create lag features ---
def create_lag_features(df, target_cols, lags=[1, 2, 3]):
    df = df.copy()
    for col in target_cols:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
    return df.dropna().reset_index(drop=True)

df_lagged = create_lag_features(df, target_cols)

# --- 5. Define features (X) and targets (Y) ---
X = df_lagged.drop(columns=['Date'] + target_cols)
Y = df_lagged[target_cols]

# --- 6. Train-test split without shuffle (time-based split) ---
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=False
)

# --- 7. Feature scaling (important for SVR & GradientBoosting) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 8. Define models ---
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    "SVR": SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
}

# --- 9. Train & evaluate models for each target ---
results = []

for target in target_cols:
    print(f"\n🎯 Target Variable: {target}")
    y_train = Y_train[target]
    y_test = Y_test[target]

    for name, model in models.items():
        # Fit model
        if name in ['SVR']:  # SVR expects 1D target
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        # Evaluate
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)

        results.append({
            'Target': target,
            'Model': name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })
        print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.3f}")

# --- 10. Compare performance ---
results_df = pd.DataFrame(results)
print("\n===== Model Performance Summary =====")
print(results_df.sort_values(by=['Target', 'RMSE']))



🎯 Target Variable: domestic passengers
LinearRegression: MAE=1092778.46, RMSE=1431994.79, R2=0.286
RandomForest: MAE=1080391.08, RMSE=1462784.02, R2=0.255
GradientBoosting: MAE=1335640.69, RMSE=1662460.91, R2=0.038
XGBoost: MAE=1375084.08, RMSE=1680337.31, R2=0.017
SVR: MAE=6275070.22, RMSE=6499893.56, R2=-13.710

🎯 Target Variable: international passengers
LinearRegression: MAE=757005.61, RMSE=860817.65, R2=-1.370
RandomForest: MAE=647057.02, RMSE=707803.53, R2=-0.602
GradientBoosting: MAE=612254.60, RMSE=679563.04, R2=-0.477
XGBoost: MAE=576870.24, RMSE=677682.58, R2=-0.469
SVR: MAE=1074622.01, RMSE=1153319.92, R2=-3.254

🎯 Target Variable: domestic freight(in tonne)
LinearRegression: MAE=6633.81, RMSE=8420.67, R2=-1.554
RandomForest: MAE=3896.19, RMSE=4566.10, R2=0.249
GradientBoosting: MAE=6220.27, RMSE=7670.33, R2=-1.119
XGBoost: MAE=5778.89, RMSE=6828.79, R2=-0.680
SVR: MAE=19975.67, RMSE=20666.52, R2=-14.383

🎯 Target Variable: international freight(in tonne)
LinearRegression: 